# DÉCLIC Phase 6 — les trois bases sur un seul GPU

Ce notebook entraîne SingPAD, puis AMME et Yousician dans la même session.
Les checkpoints restent séparés par base et architecture sur Drive.
Déposer les trois archives dans `Mon Drive/DECLIC/datasets/`, puis tout exécuter.
La reprise est automatique après une déconnexion. Aucun abonnement payant nécessaire.
Les anciennes sorties SingPAD Phase 5 sont invalidées et ne sont pas réutilisées.


In [ ]:
from pathlib import Path
import subprocess, sys, os, json, shutil, hashlib
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCE = 'singpad'
EPOCHS = 12
SEED = 42
BATCH_SIZE = 16
# Changing architecture/seed uses a separate output folder. Resuming preserves the split.
MODELS = ['dkt', 'enhanced_gru', 'item_attention']
REPO_URL = 'https://github.com/MoallaMelek/Declics-Knowledge-Tracing-for-Learner-Progress-and-Skill-Prediction..git'
REF = 'phase6/scientific-validation'
REPO_DIR = Path('/content/declic-axis7')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Existing checkout retained for reproducible resume.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c',
                str(REPO_DIR / 'configs/colab-constraints.txt'), '-e', str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
import torch
from axis7.cloud_training import SOURCE_HASHES
RUN_NAMESPACE = 'phase6-' + hashlib.sha256(json.dumps(SOURCE_HASHES, sort_keys=True).encode()).hexdigest()[:12]
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print('CPU disponible. Le GPU gratuit est facultatif, selon disponibilité Colab.')
print('Code revision:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
from google.colab import drive, files
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive', timeout_ms=600000)
    except ValueError as error:
        if not str(error).startswith('mount failed'):
            raise
        print('Reprise du montage Drive après une erreur temporaire.')
        drive.mount('/content/drive', force_remount=True, timeout_ms=600000)

DRIVE_ROOT = Path('/content/drive/MyDrive/DECLIC')
DATA_DIR = DRIVE_ROOT / 'datasets'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_NAME = 'singKT-dataset-main.zip'
drive_archive = DATA_DIR / ARCHIVE_NAME
if not drive_archive.is_file():
    print('Importer uniquement', ARCHIVE_NAME, '(conservé dans Drive pour les prochains lancements).')
    uploaded = files.upload()
    if ARCHIVE_NAME not in uploaded:
        raise FileNotFoundError('Le nom du fichier doit être ' + ARCHIVE_NAME)
    shutil.copyfile(ARCHIVE_NAME, drive_archive)
    del uploaded
LOCAL_ARCHIVE = Path('/content') / ARCHIVE_NAME
shutil.copyfile(drive_archive, LOCAL_ARCHIVE)
RUN_ROOT = DRIVE_ROOT / RUN_NAMESPACE / SOURCE / f'seed-{SEED}'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print('Données prêtes. Checkpoints et métriques :', RUN_ROOT)

## Entraîner et reprendre
Les apprenants d'entraînement, validation et test sont séparés. Chaque prévision
utilise seulement les groupes chronologiquement antérieurs. Pour SingPAD, toutes
les notes du même enregistrement et les enregistrements simultanés sont cachés
ensemble. AMME prédit le score du prochain bloc ; Yousician le prochain jour actif.
Les scores continus ne sont pas transformés artificiellement en « correct/incorrect ».
La référence est simplement la moyenne d'entraînement, sans modèle classique entraîné.

In [ ]:
from axis7.cloud_training import train, atomic_json
reports = {}
for name in MODELS:
    folder = RUN_ROOT / name
    reports[name] = train(SOURCE, LOCAL_ARCHIVE, folder, model_name=name,
                          epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
    print(name, json.dumps(reports[name]['test'], ensure_ascii=False))
key = 'log_loss' if SOURCE == 'singpad' else 'mae'
# Select on validation only, never on test results.
selected = min(reports, key=lambda name: reports[name]['validation'][key])
atomic_json(RUN_ROOT / 'selection.json', {'selected': selected, 'criterion': 'validation_' + key})
print('Architecture sélectionnée sur validation :', selected)
print('Contrôle initial sur la moyenne constante :', reports[selected]['validation_gate']['accepted'])
print('La comparaison historique ci-dessous reste obligatoire pour promouvoir SingPAD.')

In [ ]:
# Small portable package: weights, exact config, actual metrics and history.
# No raw observations, prepared cache or optimizer checkpoints exported.
# Split provenance is counts/digests; learner IDs remain in private resume files.
import zipfile
EXPORT = RUN_ROOT / f'{SOURCE}-benchmark-bundle.zip'
folder = RUN_ROOT / selected
with zipfile.ZipFile(EXPORT, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('model.pt', 'metrics.json', 'metrics.csv', 'history.json'):
        z.write(folder / name, name)
print('Export sauvegardé :', EXPORT)
print('Rapport :', folder / 'metrics.json')
# Optional download to this computer. Drive already holds the persistent copy.
# files.download(str(EXPORT))

## AMME et Yousician, sur le même GPU
Les sorties de chaque base restent séparées. Les écritures Drive sont transférées avant la fin.

In [ ]:
# Reuse this GPU for the other sources: no additional Colab session required.
from axis7.cloud_training import train, atomic_json
import zipfile, json, shutil
REMAINING = {'amme': '2gvty-osfstorage-archive.zip',
             'yousician': 'yousician_ukulele.json.zip'}
for next_source, archive_name in REMAINING.items():
    local_archive = Path('/content') / archive_name
    shutil.copyfile(DATA_DIR / archive_name, local_archive)
    source_root = DRIVE_ROOT / RUN_NAMESPACE / next_source / f'seed-{SEED}'
    model_folder = source_root / 'enhanced_gru'
    result = train(next_source, local_archive, model_folder, model_name='enhanced_gru',
                   epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED)
    atomic_json(source_root / 'selection.json', {'selected': 'enhanced_gru', 'criterion': 'validation_mae'})
    export_path = source_root / f'{next_source}-benchmark-bundle.zip'
    with zipfile.ZipFile(export_path, 'w', zipfile.ZIP_DEFLATED) as z:
        for name in ('model.pt', 'metrics.json', 'metrics.csv', 'history.json'):
            z.write(model_folder / name, name)
    print(json.dumps({'source': next_source, 'test': result['test'],
                      'validation_gate': result['validation_gate']}, ensure_ascii=False))
    print('Export sauvegardé :', export_path)
print('AMME et Yousician terminés ; l’analyse et le transfert Drive suivent.')

In [ ]:
# Matched controls, learner bootstrap, and reproducible execution environment.
import importlib.util, zipfile
spec = importlib.util.spec_from_file_location('phase6_runner', REPO_DIR / 'scripts/run_phase6_experiments.py')
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
runner.run(SOURCE, LOCAL_ARCHIVE, RUN_ROOT.parent, [SEED], False)
for name in MODELS:
    folder = RUN_ROOT / name
    export = RUN_ROOT / f'{SOURCE}-{name}-phase6-bundle.zip'
    with zipfile.ZipFile(export, 'w', zipfile.ZIP_DEFLATED) as z:
        for filename in ('model.pt', 'metrics.json', 'metrics.csv', 'history.json', 'analysis.json'):
            z.write(folder / filename, filename)
    print('Export avec comparaisons :', export)
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(RUN_ROOT / 'execution-requirements.txt').write_text(freeze)
drive.flush_and_unmount(timeout_ms=300000)
print('PHASE6_EXPORTS_COMPLETED : modèles, analyses et écritures Drive transférées.')

## Retour dans DÉCLIC
Le fichier exporté peut être téléchargé puis importé avec
`python -m axis7.musical_registry import --bundle CHEMIN.zip`.
L'import vérifie le checksum, le découpage, les paramètres et le seuil de validation.
Un modèle refusé reste consultable en recherche, sans être utilisé pour des prévisions.
Les endpoints de recherche exposent uniquement le domaine et la cible réellement
évalués. Les auto-évaluations du carnet piano ne deviennent pas des labels de réussite.